# 03 Stress Tests

Purpose:
- Run predefined stress scenarios
- Compare scenario-level returns, drawdowns, and recovery
- Inspect PnL attribution under stress


In [ ]:
from pathlib import Path
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Ensure all project-relative IO uses repository root
os.chdir(ROOT)

from src.utils.io import load_config
from src.experiments.run_baseline import run as run_baseline
from src.experiments.run_stress import run as run_stress


In [ ]:
CONFIG_PATH = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG_PATH)

backtests = run_baseline(str(CONFIG_PATH))
stress = run_stress(str(CONFIG_PATH))
stress = stress.sort_values(["pair", "scenario"]).reset_index(drop=True)
stress


In [ ]:
for pair_name, group in stress.groupby("pair"):
    print(f"{pair_name}")
    display(group[["scenario", "start", "end", "window_return", "max_drawdown", "recovery_days", "avg_gross"]])


In [ ]:
for _, row in stress.iterrows():
    pair = row["pair"]
    start = pd.Timestamp(row["start"], tz="UTC")
    end = pd.Timestamp(row["end"], tz="UTC")

    bt = backtests[pair]
    window = bt.loc[(bt.index >= start) & (bt.index <= end)].copy()
    if window.empty:
        continue

    window["equity_norm"] = window["equity"] / window["equity"].iloc[0]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(window.index, window["equity_norm"], lw=1.6)
    axes[0].set_title(f"{pair} | {row['scenario']} | equity")
    axes[0].grid(alpha=0.2)

    axes[1].plot(window.index, window["drawdown"], color="crimson", lw=1.3)
    axes[1].fill_between(window.index, window["drawdown"], 0, color="salmon", alpha=0.35)
    axes[1].set_title(f"{pair} | {row['scenario']} | drawdown")
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


In [ ]:
attrib_rows = []
for _, row in stress.iterrows():
    pair = row["pair"]
    bt = backtests[pair]
    start = pd.Timestamp(row["start"], tz="UTC")
    end = pd.Timestamp(row["end"], tz="UTC")
    window = bt.loc[(bt.index >= start) & (bt.index <= end)]
    if window.empty:
        continue

    attrib_rows.append({
        "pair": pair,
        "scenario": row["scenario"],
        "spot_pnl_mean": window["spot_pnl"].mean(),
        "lev_pnl_mean": window["lev_pnl"].mean(),
        "cost_total_mean": window["cost_total"].mean(),
    })

attrib = pd.DataFrame(attrib_rows)
attrib


In [ ]:
assert not stress.empty, "Stress table is empty"
assert (stress["max_drawdown"] <= 0).all(), "Stress max_drawdown should be <= 0"
print("Stress checks passed.")


In [ ]:
mc_summary_path = ROOT / "reports" / "tables" / "stress_monte_carlo_summary.csv"
if mc_summary_path.exists():
    mc_summary = pd.read_csv(mc_summary_path)
    mc_summary
else:
    print("Monte Carlo summary not found yet.")


In [ ]:
for pair in ["btc_bitx", "qqq_tqqq"]:
    p = ROOT / "reports" / "tables" / f"{pair}_mc_path_metrics.csv"
    if not p.exists():
        continue
    df = pd.read_csv(p)
    if df.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df["total_return"], bins=40, alpha=0.75)
    ax.axvline(df["total_return"].quantile(0.05), color="crimson", ls="--", label="5th pct")
    ax.axvline(df["total_return"].quantile(0.5), color="black", ls="--", label="median")
    ax.set_title(f"{pair}: Monte Carlo total return distribution")
    ax.set_xlabel("Total return")
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()
